In [1]:
from pathlib import Path
from time import perf_counter
from IPython.display import display, HTML

from qdk import Context, qsharp
from qdk.qre import SequentialCountingProfiler, TemporalProfiler, TemporalProfilerConfig, estimate
from qdk.qre.models import GateBased, RoundBasedFactory, SurfaceCode, TwoDimensionalYokedSurfaceCode
from qdk.qre.application import QSharpApplication

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "library/arithmetic/qsharp.json").is_file()
)
QIR_DIR = REPO_ROOT / "source/qep/tmp"
PROFILE_DIR = REPO_ROOT / "source/qep/profile_viewer/profiles"

# Creates default config.
def config(record_trace=False, aggregation_window_sec=1.0,t_factories_count=10):
    return TemporalProfilerConfig(
        clifford_gate_time_nanos=5000,
        measurment_time_nanos=1000,    
        seed=42,
        record_trace=record_trace,
        aggregation_window_nanos = int(aggregation_window_sec*1e9),
        t_factories_count=t_factories_count
    )

def report_result(result, name, t0=None):
    if t0 is not None:
        dt = perf_counter() - t0
        count = result.totals.total_operations_count
        ns_per_op = int(1e9 * dt / count)    
        print(f"Profiling time {dt:.2f}s, {ns_per_op} ns/op")

    print(f"Estimated runtime: {result.totals.total_runtime_sec}s")
    print("Logical qubits:", result.totals.logical_qubits)
    print("  of them, logical compute qubits:", result.totals.logical_compute_qubits)
    print("Physical qubits:", result.totals.physical_qubits)
    
    json_path = PROFILE_DIR / (name + ".json")
    result.to_json(json_path)
    print(f"Profile saved to {json_path}")
    display(HTML(f'<a href="http://localhost:3000/source/qep/profile_viewer/?profile_name={name}">View profile</a>'))

## Example 1 - CCNOT gate

This example has one CCNOT gate that is transpiled into Clifford+T gates (using 7 T or T-adjoint gates).
Then we use T factories to generate T states for T/T-adjoint gates.

We run porfiler with 1 factory and see how waiting for next T state becomes a bottleneck.

Then we run profiler with 5 factories and see how states form different factories are used and T gate is executed faster.

In [2]:
qsharp.init(target_profile=qsharp.TargetProfile.Base)
app_example1 = QSharpApplication("""{
    use qs = Qubit[3];
    H(qs[0]);
    H(qs[1]);
    CCNOT(qs[0], qs[1], qs[2]);
    [MResetZ(qs[0]), MResetZ(qs[1]), MResetZ(qs[2])]  
}""")

profiler = TemporalProfiler(config(record_trace=True,t_factories_count=1))
profiler.add_adapter("transpiler")
result = app_example1.profile(profiler, use_qir=True)
report_result(result, f"example1_1factory")

Estimated runtime: 0.000468s
Logical qubits: 8
  of them, logical compute qubits: 3
Physical qubits: 392
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/example1_1factory.json


In [3]:
profiler = TemporalProfiler(config(record_trace=True,t_factories_count=5))
profiler.add_adapter("transpiler")
result = app_example1.profile(profiler, use_qir=True)
report_result(result, f"example1_5factories")

Estimated runtime: 0.000154s
Logical qubits: 28
  of them, logical compute qubits: 3
Physical qubits: 1372
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/example1_5factories.json


## Example 2 - Grover search (from Q# samples)

In [4]:
ctx1=Context(target_profile=qsharp.TargetProfile.Base)
with open(REPO_ROOT / "samples/algorithms/Grover.qs") as f:
    ctx1.eval(f.read())
ctx1.eval("""
    operation GroverSearchEntryPoint(nQubits: Int) : Result[] {
        let nIterations = IterationsToMarked(nQubits);
        GroverSearch(nQubits, nIterations, ReflectAboutMarked)
    }
""")
 
n=6
app_grover = QSharpApplication(ctx1.code.GroverSearchEntryPoint, args=(n,))

t0 = perf_counter()
profiler = TemporalProfiler(config(record_trace=True, aggregation_window_sec=0.1))
profiler.add_adapter("transpiler")
result = app_grover.profile(profiler, use_qir=True)
report_result(result, f"grover_{n}", t0=t0)

Profiling time 0.07s, 18934 ns/op
Estimated runtime: 0.005307s
Logical qubits: 61
  of them, logical compute qubits: 11
Physical qubits: 2989
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/grover_6.json


In [5]:
n=16
app_grover = QSharpApplication(ctx1.code.GroverSearchEntryPoint, args=(n,))

t0 = perf_counter()
profiler = TemporalProfiler(config())
profiler.add_adapter("transpiler")
result = app_grover.profile(profiler, use_qir=True)
report_result(result, f"grover_{n}", t0=t0)

Profiling time 0.69s, 1615 ns/op
Estimated runtime: 0.553131s
Logical qubits: 81
  of them, logical compute qubits: 31
Physical qubits: 3969
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/grover_16.json


## Example 3 - modular multiplication

In [6]:
# ModMul search example.
ctx2=Context(project_root=str(REPO_ROOT / "library/arithmetic"),
            target_profile=qsharp.TargetProfile.Adaptive)
ctx2.eval("""
    operation ProfileModMul(n: Int) : Unit {
        let modulus = (1L <<< n) - 1L;
        use xs = Qubit[n];
        use ys = Qubit[n];
        use ans = Qubit[n];
        ModMul.ModMul(xs, ys, ans, modulus);
    }
""")

n=8
app_modmul = QSharpApplication(ctx2.code.ProfileModMul, args=(n,))

profiler = TemporalProfiler(config())
profiler.add_adapter("transpiler")
t0 = perf_counter()
result = app_modmul.profile(profiler, use_qir=True)
report_result(result, f"modmul_{n}", t0=t0)

Profiling time 0.14s, 6678 ns/op
Estimated runtime: 0.04548s
Logical qubits: 83
  of them, logical compute qubits: 33
Physical qubits: 4067
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/modmul_8.json


In [7]:
n=200
app_modmul = QSharpApplication(ctx2.code.ProfileModMul, args=(n,))

profiler = TemporalProfiler(config())
profiler.add_adapter("transpiler")
t0 = perf_counter()
result = app_modmul.profile(profiler, use_qir=True)
report_result(result, f"modmul_{n}", t0=t0)

Profiling time 1.46s, 90 ns/op
Estimated runtime: 34.072536s
Logical qubits: 851
  of them, logical compute qubits: 801
Physical qubits: 41699
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/modmul_200.json


## Example 4 - Shor's algorithm (50 bit) from precompiled QIR file

In [8]:
qir_path = str(QIR_DIR / "shors_50.ll")
profiler = TemporalProfiler(config())
profiler.add_adapter("transpiler")
t0 = perf_counter()
result = profiler.profile_qir(qir_path)
report_result(result, f"shors_50", t0=t0)

Profiling time 12.89s, 95 ns/op
Estimated runtime: 279.924276s
Logical qubits: 301
  of them, logical compute qubits: 251
Physical qubits: 14749
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/shors_50.json


## Example 5 - VQE energy evaluation

Profile one VQE energy-evaluation workload adapted from
[Variational Quantum Algorithms](../../../samples/chemistry/Variational%20Quantum%20Algorithms.ipynb).
This uses the sample's seven-term, two-qubit Hamiltonian and Pauli-rotation ansatz,
with 100 shots per term. Angles are chosen as multiples of pi/8 so the default
transpiler can synthesize them. The profiler models execution costs with random
measurement results; it does not compute a physically meaningful energy or run
the classical optimization loop.

In [9]:
qsharp.init(target_profile=qsharp.TargetProfile.Adaptive)
app_vqe = QSharpApplication("""
{
    let terms = [
        (-3.0, [PauliI, PauliZ]),
        (0.5, [PauliX, PauliI]),
        (1.5, [PauliX, PauliX]),
        (-0.5, [PauliX, PauliZ]),
        (-1.5, [PauliY, PauliY]),
        (2.5, [PauliZ, PauliI]),
        (-1.5, [PauliZ, PauliZ])
    ];
    mutable energy = 0.0;
    for (coefficient, bases) in terms {
        mutable total = 0.0;
        for _ in 1..100 {
            use register = Qubit[2];
            // Exp(ZZ, pi/8) and Exp(XY, -pi/8), decomposed for this backend.
            CNOT(register[0], register[1]);
            Rz(-Std.Math.PI() / 4.0, register[1]);
            CNOT(register[0], register[1]);
            within {
                H(register[0]);
                Adjoint S(register[1]);
                H(register[1]);
            } apply {
                CNOT(register[0], register[1]);
                Rz(Std.Math.PI() / 4.0, register[1]);
                CNOT(register[0], register[1]);
            }
            let measured = Measure(bases, register);
            if measured == Zero {
                total += 1.0;
            } else {
                total -= 1.0;
            }
            ResetAll(register);
        }
        energy += coefficient * total / 100.0;
    }
    energy
}
""")

profiler = TemporalProfiler(config(record_trace=True))
profiler.add_adapter("transpiler")
t0 = perf_counter()
result = app_vqe.profile(profiler, use_qir=True)
report_result(result, "vqe", t0=t0)

Profiling time 0.11s, 6234 ns/op
Estimated runtime: 0.05085s
Logical qubits: 53
  of them, logical compute qubits: 3
Physical qubits: 2597
Profile saved to /home/dfedoriaka/qdk/qep/source/qep/profile_viewer/profiles/vqe.json
